### Verificación de las cuestiones

In [2]:
# Revisando que todo esté en orden
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
pd.set_option('display.max_columns',34)

print(f"pandas ready     {pd.__version__}")
print(f"matplotlib ready {plt.matplotlib.__version__}")
print(f"seaborn ready    {sns.__version__}")
print(f"scikit-learn ready {sklearn.__version__}")
print("\nEso es todo")

pandas ready     3.0.1
matplotlib ready 3.10.8
seaborn ready    0.13.2
scikit-learn ready 1.8.0

Eso es todo


### Cargar dataset 

In [3]:
# use a subset of columns, for simplicity and speed
columns = ['question_id', 'title', 'body', 'tags', 'tag_count',
       'programming_language', 'categories', 'creation_date', 'creation_year',
       'creation_month', 'creation_weekday', 'last_activity_date',
       'view_count', 'score', 'answer_count', 'comment_count',
       'favorite_count', 'is_answered', 'has_accepted_answer',
       'accepted_answer_score', 'has_code', 'code_block_count',
       'title_word_count', 'title_char_count', 'body_word_count',
       'body_char_count', 'difficulty_score', 'quality_score',
       'owner_reputation', 'owner_badge_count', 'first_response_time_seconds',
       'first_response_time_hours', 'top_answer_score',
       'top_answer_body_length']

# Carga del dataset de kaggle
df = pd.read_csv("../dataset/stackoverflow_combined.csv", index_col="question_id", parse_dates=["creation_date"], usecols=columns)

print(f"Dataset listo")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

Dataset listo
Filas: 95,636
Columnas: 33


### Revisión de filas

In [ ]:
# Revisar las 10 primeras filas
df.head(10)

### Revision de columnas

In [ ]:
df.columns

### Estructura del dataset

In [ ]:
#Tipos de datos y valores nulos
df.info()

### Rows with nulls

In [ ]:
#Filas con los valores nulos
df[df.isna().any(axis=1)]

In [ ]:
# Revisar valores nulos por columna
nulos = df.isnull().sum()
nulos_porcentaje = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    "valores_nulos": nulos,
    "porcentaje": nulos_porcentaje
})

# Solo mostrar columnas que sí tienen nulos
print(resumen_nulos[resumen_nulos["valores_nulos"] > 0])

### Deleting rows with no body

In [ ]:
# Eliminar las 22 filas sin body
df = df.dropna(subset=["body"])

print(f"Filas antes:  95,636")
print(f"Filas después: {len(df):,}")
print(f"Filas eliminadas: {95636 - len(df)}")

### Unique values

In [ ]:
counts = df["programming_language"].value_counts()

print(counts.to_string())

### Estadistica general

In [ ]:
#Resumen estadistico
df.describe()

### Revisar los lenguajes disponibles

In [ ]:
# Revisar cuales son los lenguajes que aparecen
print("Lenguajes del dataset:")
print(df["programming_language"].unique())
print(f"\nTotal: {df["programming_language"].nunique()} lenguajes")

### Columnas utiles 

In [ ]:
df = df[[
#     'title', 'body', 'tags', 'tag_count', 
       'programming_language',
       #'categories', 
       'creation_date', 
       #'creation_year', 'creation_month', 'creation_weekday', 'last_activity_date', 
       'view_count', 'score', 'answer_count', 
       #'comment_count', 'favorite_count',
       'is_answered', 'has_accepted_answer',
       #'accepted_answer_score', 'has_code','code_block_count', 'title_word_count', 'title_char_count', 'body_word_count', 'body_char_count', 
       'difficulty_score', 'quality_score',
       #'owner_reputation', 'owner_badge_count','first_response_time_seconds', 'first_response_time_hours','top_answer_score', 'top_answer_body_length'
       ]].copy()

df

### Filas no deseadas


In [ ]:
no_lenguajes = ["shell", "bash", "powershell", "html", "css", "haskell", "dart"]

df = df[~df["programming_language"].isin(no_lenguajes)].copy()

# haskell y dart los quite porque nada mas hay 3 preguntas de haskell y 8 de dart

## Analisis after cleaning

### Cuantas preguntas hay por lenguaje?

In [ ]:
ax= df["programming_language"].value_counts() \
#     .plot(kind="bar", title="Preguntas por lengauje")

# ax.set_xlabel('Lenguaje')
# ax.set_ylabel('Count')

# plt.savefig("../graficos/01_preguntas_por_lenguaje.png", dpi=150)
# plt.show()

ax

### Que lenguajes tienen mas vistas

In [4]:
ax = df.groupby("programming_language")["view_count"] \
    .mean() \
    .sort_values(ascending=False) \
    # .plot(kind="bar")

# ax.set_xlabel('Lenguaje')
# ax.set_ylabel('Numero de vistas')

# plt.savefig("../graficos/vistas_por_lenguaje.png", dpi=150)
# plt.show()

ax
    

programming_language
go            498.141135
ruby          492.514032
scala         355.513257
powershell    335.045455
shell         324.979167
kotlin        322.103896
perl          316.121951
bash          293.951613
matlab        273.806110
rust          271.742315
php           258.705112
typescript    241.455059
swift         236.765860
css           236.635598
python        204.949777
c++           180.828528
javascript    176.384228
java          176.061029
c             171.052988
c#            133.699632
haskell       130.666667
sql           130.273953
html          114.523438
dart          106.500000
r             105.397886
Name: view_count, dtype: float64

### Puntaje promedio por lenguaje

In [ ]:
ax = df.groupby("programming_language")["score"] \
    .mean() \
    .sort_values(ascending=False) \
    .plot(kind="bar")

ax.set_xlabel('Lenguaje')
ax.set_ylabel('Puntaje promedio')

### Numero de preguntas por el promedio de vistas de cada lenguaje de programación

In [ ]:
# calcula el numero de vistas por pregunta de cada lenguaje
result = df.groupby("programming_language").agg({
    "programming_language" : "count",
    "view_count" : "mean"
}).rename(columns={"programming_language" : "num_preguntas", "view_count" : "promedio_vistas"})

result["preguntas_por_vista"] = result["num_preguntas"] / result["promedio_vistas"]

result = result.sort_values("preguntas_por_vista", ascending=False)

result


In [ ]:
df.head()

In [ ]:
## Just python
python_df = df[df["programming_language"] == "python"].copy()

sns.scatterplot(data=python_df,
                x="creation_date",
                y="view_count")

sns.regplot(data=python_df,
            x="creation_date",
            y="view_count", 
            scatter=False, 
            color="red")

plt.title("Numero de vistas por pregunta a lo largo de los años")
plt.show()

